In [2]:
%run init_env.py

Added to Python path: C:\git\cs5305\genai-langchain
Environment initialization completed successfully.


## Summarize messages

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

llm_std = create_azure_llm("chat")
llm_nano = create_azure_llm("nano")

agent = create_agent(
    model=llm_std,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm_nano,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

Creating Azure OpenAI LLM
  Deployment: lums-gpt-4.1-mini
  Endpoint: https://aoai-foundry-swc.openai.azure.com/
  API Version: 2025-01-01-preview
  Temperature: 0.7
Creating Azure OpenAI LLM
  Deployment: gpt-5.4-nano
  Endpoint: https://aoai-foundry-swc.openai.azure.com/
  API Version: 2025-01-01-preview
  Temperature: 0.7


In [4]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nDetermine and answer a series of factual/fictional questions about “Lunapolis” (capital of the moon), including weather, population of cheese miners, and whether their union will strike.\n\n## SUMMARY\n- Established that the “capital of the moon” is **Lunapolis**.\n- Provided weather for Lunapolis: **clear skies**, **high 120°C**, **low -100°C**.\n- Stated that **100,000 cheese miners** live in Lunapolis.\n- Predicted the cheese miners’ union **will strike**, reasoning that they are **unhappy with the new president**.\n\n## ARTIFACTS\nNone.\n\n## NEXT STEPS\nNo further tasks remain based on the provided conversation.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='ce0ae384-46a2-4efe-94a4-2c3a1c62f63f'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, resp

In [5]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
Determine and answer a series of factual/fictional questions about “Lunapolis” (capital of the moon), including weather, population of cheese miners, and whether their union will strike.

## SUMMARY
- Established that the “capital of the moon” is **Lunapolis**.
- Provided weather for Lunapolis: **clear skies**, **high 120°C**, **low -100°C**.
- Stated that **100,000 cheese miners** live in Lunapolis.
- Predicted the cheese miners’ union **will strike**, reasoning that they are **unhappy with the new president**.

## ARTIFACTS
None.

## NEXT STEPS
No further tasks remain based on the provided conversation.


## Trim/delete messages

In [6]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [7]:
agent = create_agent(
    model=llm_std,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [8]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='fdea8df5-1133-4d96-a7ae-c2066964c1d7'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='62406218-ca10-4233-9da4-a337641ace9d', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='4010b047-d731-452a-8581-372a9dd3968b'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='12c5aa35-6f6e-4777-91ea-35c7e41e9175', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='b8822126-1f7c-4d25-876f-c324d4a52688'),
              AIMessage(content="Have you noticed if the device feels unusually hot or cold? So

In [9]:
print(response["messages"][-1].content)

Have you noticed if the device feels unusually hot or cold? Sometimes extreme temperatures can affect its ability to power on properly. If it feels very hot, it might need to cool down before trying again. If it's very cold, warming it up gently to room temperature could help. Could you check and let me know?
